![alt text](../../pythonexposed-high-resolution-logo-black.jpg "Optionele titel")

## Introductie tot pandas - Hierarchical Indexing

### Overzicht

In deze notebook behandelen we de basisprincipes van hierarchische indexering in pandas, wat een methode is om gegevens in meerdere niveaus te organiseren. Deze techniek stelt gebruikers in staat om complexe datasets te analyseren en eenvoudig te slicen of te groeperen op verschillende hiërarchische lagen.  We verkennen hoe we multi-indexen kunnen maken, gebruiken en manipuleren, en hoe aggregaties op multi-indexen worden uitgevoerd.

### **1. Een Multiply Indexed Series**

#### **1.1 Wat is een MultiIndex?**

Een MultiIndex is een index in pandas die **meerdere niveaus** bevat, wat nuttig is voor het organiseren van complexe datasets. Hiermee kun je datasets structureren op basis van meerdere dimensies, zoals categorieën en subcategorieën. Een praktische toepassing hiervan is bijvoorbeeld een dataset waarin je verkoopgegevens wilt organiseren per regio, land en stad. Met een MultiIndex kun je eenvoudig totalen berekenen voor elk niveau (bijvoorbeeld het totale aantal verkopen per land), of specifieke subsets van de data opvragen (zoals de verkoop in een specifieke stad).

#### **1.2 Voorbeeld van een Multi Indexed Series**

In [1]:
import pandas as pd
import numpy as np

# Creëer een MultiIndex voor verkoop per regio, land en stad (uitleg: zie verder)
index = pd.MultiIndex.from_product(
    [['Noord-Amerika', 'Europa'], ['VS', 'Canada', 'België', 'Nederland'], ['New York', 'Toronto', 'Brussel', 'Amsterdam']],
    names=['Regio', 'Land', 'Stad']
)
# Aantal combinaties in de MultiIndex bepalen (2*4*4)
num_combinations = len(index)
num_combinations

32

In [2]:
# Willekeurige data genereren voor alle combinaties
data = pd.Series(
    np.random.randint(50, 200, size=num_combinations),
    index=index
)

print(data)

Regio          Land       Stad     
Noord-Amerika  VS         New York     183
                          Toronto      142
                          Brussel       65
                          Amsterdam    187
               Canada     New York      69
                          Toronto      157
                          Brussel      127
                          Amsterdam     58
               België     New York     198
                          Toronto      189
                          Brussel      143
                          Amsterdam    160
               Nederland  New York     146
                          Toronto      154
                          Brussel      114
                          Amsterdam     85
Europa         VS         New York      57
                          Toronto      161
                          Brussel       84
                          Amsterdam     90
               Canada     New York     105
                          Toronto      151
                  

De data wordt gestructureerd in rijen met meerdere indexniveaus.

#### **1.3 Waarom MultiIndex gebruiken?**

- Handig bij het analyseren van data met hiërarchische relaties.
- Ondersteunt complexe slicing- en filtering-operaties.

### **2. Methoden voor het maken van een MultiIndex**

#### 2.1 Gebruik van `from_arrays`

In [3]:
arrays = [
    ['A', 'A', 'B', 'B'],
    [1, 2, 1, 2]
]
index = pd.MultiIndex.from_arrays(arrays, names=('Letter', 'Number'))
data = pd.Series([10, 20, 30, 40], index=index)

print(data)

Letter  Number
A       1         10
        2         20
B       1         30
        2         40
dtype: int64


#### 2.2 Gebruik van `from_tuples`

In [4]:
tuples = [('A', 1), ('A', 2), ('B', 1), ('B', 2)]
index = pd.MultiIndex.from_tuples(tuples, names=('Letter', 'Number'))
data = pd.Series([10, 20, 30, 40], index=index)

print(data)

Letter  Number
A       1         10
        2         20
B       1         30
        2         40
dtype: int64


#### 2.3 Gebruik van `from_product`

De functie `pandas.MultiIndex.from_product` wordt gebruikt om een MultiIndex te maken op basis van het Cartesian product van (alle mogelijke combinaties van waarden uit de ingevoerde lijsten of iterables, waarbij elke combinatie exact één keer voorkomt) meerdere arrays of iterables. Dit is handig bij het werken met hiërarchische indices in DataFrames of Series.

Met `pandas.MultiIndex.from_product` kun je een MultiIndex maken op basis van combinaties van waardes uit meerdere lijsten. Dit is vooral nuttig wanneer je data op meerdere niveaus wilt organiseren.

Belangrijkste parameters:

-`iterables`: Een lijst van iterables (zoals lijsten, arrays, of series) waarvan het Cartesian product wordt gemaakt.

-`names` (optioneel): Namen voor de verschillende niveaus van de index.

Voorbeeldgebruik:

In [5]:
import pandas as pd

# Data
levels = [['A', 'B'], [1, 2, 3]]

# MultiIndex maken
multi_index = pd.MultiIndex.from_product(levels, names=["Letter", "Number"])

# DataFrame maken met de MultiIndex
df = pd.DataFrame({"Value": [10, 20, 30, 40, 50, 60]}, index=multi_index)

print(df)

               Value
Letter Number       
A      1          10
       2          20
       3          30
B      1          40
       2          50
       3          60


### Hoe werkt het?

- De `from_product` maakt het Cartesian product van de lijsten `['A', 'B']` en `[1, 2, 3]`. Dit betekent dat alle combinaties worden gemaakt, bijvoorbeeld `('A', 1)`, `('A', 2)`, etc.

- Deze combinaties worden gebruikt om een **MultiIndex** te bouwen, die vervolgens gebruikt wordt als index voor een DataFrame.

**Toepassing in de praktijk**

Dit is vooral handig bij datasets waar je met hiërarchische gegevens werkt, bijvoorbeeld:

- Verkoopdata per regio en per productcategorie.
- Meetwaarden van verschillende sensoren en tijdstippen.
- Testresultaten van experimenten met meerdere variabelen.

In [14]:
# MultiIndex maken
index = pd.MultiIndex.from_product(
    [['A', 'B'], [1, 2]],
    names=['Letter', 'Number']
)
data = pd.Series([10, 20, 30, 40], index=index)

# Zorg ervoor dat de namen correct zijn ingesteld
data.index.names = ['Letter', 'Number']

print(data)

Letter  Number
A       1         10
        2         20
B       1         30
        2         40
dtype: int64


### 3. Indexeren en slicen van een MultiIndex

#### 3.1 Toegang tot een specifiek niveau

In [7]:
# Data ophalen voor een specifieke letter
print(data['A'])

Number
1    10
2    20
dtype: int64


#### 3.2 Slicen op meerdere niveaus

In [8]:
# Gegevens ophalen voor een specifiek niveau en subniveau
print(data.loc['A', 1])

10


#### **3.3 Gebruik van `xs` (cross-section)**

In [9]:
# Haal alle gegevens op voor het niveau 'Number' = 1
print(data.xs(1, level='Number'))

Letter
A    10
B    30
dtype: int64


### 4. Data Aggregations op Multi-Indices

#### 4.1 Gebruik van `.groupby`

In [10]:
data = data.reset_index()
grouped = data.groupby('Letter').sum()
print(grouped)

        Number   0
Letter            
A            3  30
B            3  70


#### 4.2 Gebruik van `.unstack` en `.stack`

In [21]:
# Eerst even tonen wat we gaan unstacken en restacken :-)
print(f"Initiële data: \n{data}\n")

# MultiIndex omzetten naar een DataFrame
unstacked = data.unstack(level='Number')
print(f"Unstacked data (de waarden van number zijn nu kolommen): \n{unstacked}\n")

# Data weer terugzetten naar MultiIndex
restacked = unstacked.stack()
print(f"Restacked data: \n{restacked}")

Initiële data: 
Letter  Number
A       1         10
        2         20
B       1         30
        2         40
dtype: int64

Unstacked data (de waarden van number zijn nu kolommen): 
Number   1   2
Letter        
A       10  20
B       30  40

Restacked data: 
Letter  Number
A       1         10
        2         20
B       1         30
        2         40
dtype: int64


#### **4.3 Pivot-tabellen met MultiIndex**

Een pivot-tabel transformeert data door deze samen te vatten op basis van bepaalde kolommen.

![alt text](images/pivot.png "Optionele titel")

In [23]:
# De MultiIndex wordt omgezet naar een gewone DataFrame: de functie reset_index() verplaatst de indexen van je 
# DataFrame naar kolommen, waardoor je MultiIndex-structuur wordt omgezet naar een 'platte' DataFrame.

# We vertrekken van een MultiIndex DataFrame met slechts één "waarde"-kolom, zonder naam. 
# Wanneer je een DataFrame omzet naar een platte structuur met reset_index, wil Pandas een naam geven aan die waarden.
# Door name='Value' te gebruiken, geef je een duidelijke naam aan deze kolom met waarden.
# Een kolom genaamd Value wordt dus toegevoegd waarin de oorspronkelijke waarden van de data (10,20,30,40) staan.
# Als je oorspronkelijke DataFrame meerdere kolommen had (zoals Value1 en Value2), dan is reset_index slimmer. 
# Het behoudt de namen van de kolommen (die ze ongetwijfeld reeds hadden want je had meerdere kolommen.
data_frame = data.reset_index(name='Value')

pivot_table = data_frame.pivot_table(
    values='Value',   # Dit bepaalt welke kolom moet worden samengevat (in dit geval Value).
    index='Letter',   # Dit bepaalt dat de waarden van de kolom Letter de rijenindex worden.
    columns='Number',
    aggfunc='sum'
)

print(pivot_table)

Number   1   2
Letter        
A       10  20
B       30  40
